# Notebook 4 — DINOv2 Embeddings + Unsupervised Clustering

Use DINOv2 ViT-L embeddings (extracted in Notebook 3) to:
1. Run KMeans clustering with different K values
2. Visualize clusters with UMAP and t-SNE
3. Compare discovered clusters to actual species labels
4. Quantify cluster quality (NMI, ARI, silhouette)

**Colab GPU:** Not required — this notebook runs mostly on CPU (after loading embeddings).

In [ ]:
#@title 1. Setup
from pathlib import Path
import sys

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/wild_animal_image_classification'),
    Path('/content'),
]
repo_root = None
for candidate in repo_candidates:
    if (candidate / 'configs' / 'config.py').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository root.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, confusion_matrix, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import normalize

from configs.config import CFG

CFG.ensure_output_dirs()
RESULTS_DIR = CFG.results_dir
EMBED_DIR = CFG.embeddings_dir
SPLITS_DIR = CFG.splits_dir

In [ ]:
#@title 2. Load Embeddings (extracted by Notebook 3)
d = np.load(f'{EMBED_DIR}/dinov2_train_embeddings.npz')
train_emb, train_labels = d['embeddings'], d['labels']

d = np.load(f'{EMBED_DIR}/dinov2_test_embeddings.npz')
test_emb, test_labels = d['embeddings'], d['labels']

# Combine for clustering (use all available data)
all_emb = np.concatenate([train_emb, test_emb], axis=0)
all_labels = np.concatenate([train_labels, test_labels], axis=0)

# L2 normalize embeddings (helps KMeans with cosine-like distance)
all_emb_norm = normalize(all_emb, norm='l2')

print(f'Total embeddings: {all_emb.shape}')
print(f'Unique true labels: {len(np.unique(all_labels))}')

# Load class names
with open(f'{SPLITS_DIR}/cat_names.json') as f:
    cat_names = json.load(f)
with open(f'{SPLITS_DIR}/label_to_cat.json') as f:
    label_to_cat = json.load(f)
with open(f'{SPLITS_DIR}/metadata.json') as f:
    meta = json.load(f)
NUM_CLASSES = meta['num_classes']

class_names = [cat_names.get(str(label_to_cat[str(i)]), f'class_{i}') for i in range(NUM_CLASSES)]

In [ ]:
#@title 3. KMeans Clustering with Multiple K Values
K_VALUES = [NUM_CLASSES, NUM_CLASSES * 2, 10, 20, 30]
K_VALUES = sorted(set(K_VALUES))  # deduplicate

clustering_results = {}

for k in K_VALUES:
    print(f'\nKMeans with K={k}...')
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    cluster_labels = kmeans.fit_predict(all_emb_norm)

    nmi = normalized_mutual_info_score(all_labels, cluster_labels)
    ari = adjusted_rand_score(all_labels, cluster_labels)

    # Silhouette on a subsample (full is too slow for large datasets)
    n_sample = min(10000, len(all_emb_norm))
    idx = np.random.choice(len(all_emb_norm), n_sample, replace=False)
    sil = silhouette_score(all_emb_norm[idx], cluster_labels[idx])

    clustering_results[k] = {
        'cluster_labels': cluster_labels,
        'nmi': nmi,
        'ari': ari,
        'silhouette': sil,
        'inertia': kmeans.inertia_,
    }

    print(f'  NMI: {nmi:.4f} | ARI: {ari:.4f} | Silhouette: {sil:.4f}')

In [ ]:
#@title 4. Hungarian Matching — Cluster Accuracy
def cluster_accuracy(true_labels, cluster_labels):
    """
    Use the Hungarian algorithm to find the best 1-to-1 mapping
    between cluster IDs and true class labels, then compute accuracy.
    """
    n_clusters = len(np.unique(cluster_labels))
    n_classes = len(np.unique(true_labels))
    size = max(n_clusters, n_classes)

    cost_matrix = np.zeros((size, size), dtype=np.int64)
    for c, t in zip(cluster_labels, true_labels):
        cost_matrix[c, t] += 1

    row_ind, col_ind = linear_sum_assignment(-cost_matrix)
    correct = cost_matrix[row_ind, col_ind].sum()
    return correct / len(true_labels), dict(zip(row_ind.tolist(), col_ind.tolist()))

# Compute for K = NUM_CLASSES
best_k = NUM_CLASSES
cl = clustering_results[best_k]['cluster_labels']
acc, mapping = cluster_accuracy(all_labels, cl)
print(f'\nCluster accuracy with K={best_k} (Hungarian matching): {acc:.4f}')
print(f'Cluster → Class mapping: {mapping}')

In [ ]:
#@title 5. Metrics Summary Table
print(f'\n{"="*65}')
print(f'{"K":>5} | {"NMI":>8} | {"ARI":>8} | {"Silhouette":>10} | {"Inertia":>12}')
print(f'{"-"*65}')
for k, r in sorted(clustering_results.items()):
    print(f'{k:>5} | {r["nmi"]:>8.4f} | {r["ari"]:>8.4f} | {r["silhouette"]:>10.4f} | {r["inertia"]:>12.1f}')
print(f'{"="*65}')

In [ ]:
#@title 6. UMAP Visualization
print('Computing UMAP projection (this takes a few minutes)...')

# Subsample for visualization if dataset is large
MAX_VIS = 20000
if len(all_emb_norm) > MAX_VIS:
    vis_idx = np.random.choice(len(all_emb_norm), MAX_VIS, replace=False)
else:
    vis_idx = np.arange(len(all_emb_norm))

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
umap_coords = reducer.fit_transform(all_emb_norm[vis_idx])

# Plot: True labels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

scatter1 = ax1.scatter(umap_coords[:, 0], umap_coords[:, 1],
                       c=all_labels[vis_idx], cmap='tab20', s=3, alpha=0.5)
ax1.set_title('UMAP — True Species Labels', fontsize=14)
ax1.set_xlabel('UMAP 1'); ax1.set_ylabel('UMAP 2')
cbar1 = plt.colorbar(scatter1, ax=ax1)
cbar1.set_label('Species Label')

# Plot: KMeans clusters
cl_vis = clustering_results[best_k]['cluster_labels'][vis_idx]
scatter2 = ax2.scatter(umap_coords[:, 0], umap_coords[:, 1],
                       c=cl_vis, cmap='tab20', s=3, alpha=0.5)
ax2.set_title(f'UMAP — KMeans Clusters (K={best_k})', fontsize=14)
ax2.set_xlabel('UMAP 1'); ax2.set_ylabel('UMAP 2')
cbar2 = plt.colorbar(scatter2, ax=ax2)
cbar2.set_label('Cluster ID')

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/umap_clusters_vs_true.png', dpi=150)
plt.show()

In [ ]:
#@title 7. t-SNE Visualization
print('Computing t-SNE (may take 5-10 min)...')

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_coords = tsne.fit_transform(all_emb_norm[vis_idx])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

scatter1 = ax1.scatter(tsne_coords[:, 0], tsne_coords[:, 1],
                       c=all_labels[vis_idx], cmap='tab20', s=3, alpha=0.5)
ax1.set_title('t-SNE — True Species Labels', fontsize=14)
ax1.set_xlabel('t-SNE 1'); ax1.set_ylabel('t-SNE 2')

cl_vis = clustering_results[best_k]['cluster_labels'][vis_idx]
scatter2 = ax2.scatter(tsne_coords[:, 0], tsne_coords[:, 1],
                       c=cl_vis, cmap='tab20', s=3, alpha=0.5)
ax2.set_title(f't-SNE — KMeans Clusters (K={best_k})', fontsize=14)
ax2.set_xlabel('t-SNE 1'); ax2.set_ylabel('t-SNE 2')

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/tsne_clusters_vs_true.png', dpi=150)
plt.show()

In [ ]:
#@title 8. UMAP with Named Class Legend
fig, ax = plt.subplots(figsize=(14, 10))

for label_id in range(NUM_CLASSES):
    mask = all_labels[vis_idx] == label_id
    if mask.sum() > 0:
        ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                   s=5, alpha=0.5, label=class_names[label_id])

ax.legend(markerscale=5, fontsize=9, loc='upper right', ncol=2)
ax.set_title('DINOv2 Embeddings — UMAP (True Species Labels)', fontsize=14)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/umap_true_labels_legend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#@title 9. Elbow Plot & Metrics vs K
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ks = sorted(clustering_results.keys())
nmis = [clustering_results[k]['nmi'] for k in ks]
aris = [clustering_results[k]['ari'] for k in ks]
sils = [clustering_results[k]['silhouette'] for k in ks]
inertias = [clustering_results[k]['inertia'] for k in ks]

axes[0].plot(ks, inertias, 'bo-')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Plot'); axes[0].grid(alpha=0.3)

axes[1].plot(ks, nmis, 'rs-', label='NMI')
axes[1].plot(ks, aris, 'g^-', label='ARI')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
axes[1].set_title('Cluster Quality vs K'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ks, sils, 'mD-')
axes[2].set_xlabel('K'); axes[2].set_ylabel('Silhouette Score')
axes[2].set_title('Silhouette vs K'); axes[2].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/clustering_metrics_vs_k.png', dpi=150)
plt.show()

In [ ]:
#@title 10. Cross-Tab: Cluster vs True Label
cl = clustering_results[best_k]['cluster_labels']

ct = pd.crosstab(
    pd.Series(cl, name='Cluster'),
    pd.Series(all_labels, name='True Label')
)
ct.columns = [class_names[i] for i in ct.columns]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title(f'Cluster × Species Cross-Tab (K={best_k})', fontsize=14)
ax.set_ylabel('Cluster ID'); ax.set_xlabel('True Species')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/cluster_species_crosstab.png', dpi=150)
plt.show()

In [ ]:
#@title 11. Save Clustering Results
cluster_summary = {}
for k, r in clustering_results.items():
    cluster_summary[str(k)] = {
        'nmi': float(r['nmi']),
        'ari': float(r['ari']),
        'silhouette': float(r['silhouette']),
        'inertia': float(r['inertia']),
    }

cluster_summary['cluster_accuracy_hungarian'] = float(acc)
cluster_summary['best_k'] = best_k

with open(f'{RESULTS_DIR}/clustering_results.json', 'w') as f:
    json.dump(cluster_summary, f, indent=2)

print(f'Clustering results saved to {RESULTS_DIR}/clustering_results.json')
print('\nDone!')